In [6]:
########  0000000 ######## 

import csv
import json
import os
import urllib.request
 
URL = "https://api.frankfurter.app/2026-01-01..2026-09-01?from=EUR&to=USD"
CHEMIN_CSV = "donnees/taux.csv"
CHEMIN_CACHE = "cache/reponse.json"

def telecharger(url, chemin_cache):
    """Appelle l'API, sauvegarde la réponse brute et renvoie un dictionnaire."""
    requete = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(requete) as reponse:
        contenu = reponse.read()
    os.makedirs(os.path.dirname(chemin_cache), exist_ok=True)
    with open(chemin_cache, "wb") as f:
        f.write(contenu)
    return json.loads(contenu)
 
 
def ecrire_csv(data, chemin):
    """Écrit les taux du dictionnaire dans un CSV à deux colonnes (date, USD)."""
    os.makedirs(os.path.dirname(chemin), exist_ok=True)
    with open(chemin, "w", encoding="utf-8", newline="") as f:
        ecrivain = csv.writer(f)
        ecrivain.writerow(["date", "USD"])
        for date in sorted(data["rates"]):
            ecrivain.writerow([date, data["rates"][date]["USD"]])



In [11]:
########  A.1  ######## 


def lire_taux(chemin):
    """Lit le CSV et renvoie une liste de couples (date, taux)."""
    donnees = []
    with open(chemin, encoding="utf-8") as f:
        lecteur = csv.reader(f)
        next(lecteur)  # saute l'en-tête
        for date, taux in lecteur:
            donnees.append((date, float(taux)))
    return donnees


def moyenne(taux):
    """Renvoie la moyenne d'une liste de taux."""
    return sum(taux) / len(taux)
 
 
def min_max(taux):
    """Renvoie le couple (taux minimum, taux maximum)."""
    return min(taux), max(taux)


"""On vérifie si le fichier existe déjà, et si pas le cas on le récupère via l'API"""
if not os.path.exists(CHEMIN_CSV):
    data = telecharger(URL, CHEMIN_CACHE)
    ecrire_csv(data, CHEMIN_CSV)
    print("CSV recréé depuis l'API")
else:
    print("CSV déjà présent")


"""On lit les données dans le csv, on obtient une liste de tuple avec (date, taux)"""
data = lire_taux(CHEMIN_CSV)
print(data)



"""On récupère uniquement les taux dans "data", """
taux = []
for date, t in data:
    taux.append(t)

moy = moyenne(taux)
moy = round(moy,3)
print("la moyenne est ", moy)

max, min = min_max(taux)
print("le maximum est ", max)
print("le minimum est ", min)



CSV déjà présent
[('2025-12-31', 1.175), ('2026-01-02', 1.1721), ('2026-01-05', 1.1664), ('2026-01-06', 1.1707), ('2026-01-07', 1.1684), ('2026-01-08', 1.1675), ('2026-01-09', 1.1642), ('2026-01-12', 1.1692), ('2026-01-13', 1.1654), ('2026-01-14', 1.1651), ('2026-01-15', 1.1624), ('2026-01-16', 1.1617), ('2026-01-19', 1.1631), ('2026-01-20', 1.1728), ('2026-01-21', 1.1739), ('2026-01-22', 1.1706), ('2026-01-23', 1.1742), ('2026-01-26', 1.1836), ('2026-01-27', 1.1929), ('2026-01-28', 1.1974), ('2026-01-29', 1.1968), ('2026-01-30', 1.1919), ('2026-02-02', 1.184), ('2026-02-03', 1.1801), ('2026-02-04', 1.182), ('2026-02-05', 1.1798), ('2026-02-06', 1.1794), ('2026-02-09', 1.1886), ('2026-02-10', 1.1894), ('2026-02-11', 1.19), ('2026-02-12', 1.1874), ('2026-02-13', 1.1862), ('2026-02-16', 1.1855), ('2026-02-17', 1.1826), ('2026-02-18', 1.1845), ('2026-02-19', 1.1753), ('2026-02-20', 1.1767), ('2026-02-23', 1.1784), ('2026-02-24', 1.1777), ('2026-02-25', 1.1784), ('2026-02-26', 1.1814), ('2

In [12]:
########  A.2  ######## 

import math
import statistics
 
m_perso = moyenne(taux)
m_stat = statistics.mean(taux)

print("Ma fonction      ", m_perso)
print("statistics.mean  ", m_stat)
print("Égalité stricte  ", m_perso == m_stat)
print("math.isclose     ", math.isclose(m_perso, m_stat))

Ma fonction       1.1623684210526315
statistics.mean   1.1623684210526315
Égalité stricte   True
math.isclose      True


In [13]:
########  B.1  ######## 

import urllib.error
 
 
def telecharger_securise(url, chemin_cache):
    """Comme telecharger, mais affiche un message clair et renvoie None si l'appel échoue."""
    requete = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    try:
        with urllib.request.urlopen(requete, timeout=10) as reponse:
            contenu = reponse.read()
        data = json.loads(contenu)
    except urllib.error.HTTPError as e:      # avant URLError, dont elle hérite
        print(f"Erreur HTTP {e.code} ({e.reason})")
        return None
    except urllib.error.URLError as e:
        print(f"Serveur injoignable ou pas de connexion ({e.reason})")
        return None
    except TimeoutError:
        print("Pas de réponse du serveur après 10 secondes")
        return None
    except json.JSONDecodeError:
        print("Réponse illisible, ce n'est pas du JSON valide")
        return None
    else:                                    # seulement si tout s'est bien passé
        os.makedirs(os.path.dirname(chemin_cache), exist_ok=True)
        with open(chemin_cache, "wb") as f:
            f.write(contenu)
        return data
    finally:                                 # dans tous les cas
        print("Fin de l'appel à", url)

# Un test par cas
urls_test = [
    URL,                                         # cas normal
    "https://api.frankfurter.app/n-existe-pas",  # code HTTP d'erreur
    "https://domaine-inexistant.invalid/",       # pas de connexion
    "https://example.com",                       # réponse qui n'est pas du JSON
]
for url_test in urls_test:
    resultat = telecharger_securise(url_test, "cache/test.json")
    print("Résultat", "données reçues" if resultat is not None else None, "\n")

Fin de l'appel à https://api.frankfurter.app/2026-01-01..2026-09-01?from=EUR&to=USD
Résultat données reçues 

Erreur HTTP 404 (Not Found)
Fin de l'appel à https://api.frankfurter.app/n-existe-pas
Résultat None 

Serveur injoignable ou pas de connexion ([Errno 8] nodename nor servname provided, or not known)
Fin de l'appel à https://domaine-inexistant.invalid/
Résultat None 

Réponse illisible, ce n'est pas du JSON valide
Fin de l'appel à https://example.com
Résultat None 



In [14]:
########  B.2  ######## 

import logging
 
CHEMIN_LOG = "logs/seance2.log"
os.makedirs(os.path.dirname(CHEMIN_LOG), exist_ok=True)
 
logging.basicConfig(
    level=logging.INFO,                      # DEBUG est ignoré, INFO et au-dessus sont gardés
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(CHEMIN_LOG, encoding="utf-8"),  # écrit dans le fichier .log
        logging.StreamHandler(),                            # affiche aussi à l'écran
    ],
    force=True,                              # remplace une configuration déjà faite (Jupyter)
)
logger = logging.getLogger("seance2")
 
 
def telecharger_journalise(url, chemin_cache):
    """Comme telecharger_securise, mais journalise au lieu d'afficher avec print."""
    logger.info(f"Appel de l'API {url}")
    requete = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    try:
        with urllib.request.urlopen(requete, timeout=10) as reponse:
            contenu = reponse.read()
        data = json.loads(contenu)
    except urllib.error.HTTPError as e:
        logger.error(f"Erreur HTTP {e.code} ({e.reason})")
        return None
    except urllib.error.URLError as e:
        logger.error(f"Serveur injoignable ou pas de connexion ({e.reason})")
        return None
    except TimeoutError:
        logger.error("Pas de réponse du serveur après 10 secondes")
        return None
    except json.JSONDecodeError:
        logger.error("Réponse illisible, ce n'est pas du JSON valide")
        return None
    else:
        os.makedirs(os.path.dirname(chemin_cache), exist_ok=True)
        with open(chemin_cache, "wb") as f:
            f.write(contenu)
        if not data.get("rates"):
            logger.warning("Réponse reçue mais elle ne contient aucun taux")
        else:
            logger.info(f"{len(data['rates'])} dates reçues, réponse sauvegardée")
        return data
    finally:
        logger.info("Fin de l'appel")
 
 
# Un test par niveau de log
urls_test = [
    URL,                                         # info, cas normal
    "https://api.frankfurter.app/currencies",    # warning, JSON valide mais sans taux
    "https://api.frankfurter.app/n-existe-pas",  # error, code HTTP d'erreur
    "https://domaine-inexistant.invalid/",       # error, pas de connexion
    "https://example.com",                       # error, réponse qui n'est pas du JSON
]
for url_test in urls_test:
    telecharger_journalise(url_test, "cache/test.json")


2026-09-16 15:55:17,316 | INFO | Appel de l'API https://api.frankfurter.app/2026-01-01..2026-09-01?from=EUR&to=USD
2026-09-16 15:55:17,817 | INFO | 171 dates reçues, réponse sauvegardée
2026-09-16 15:55:17,818 | INFO | Fin de l'appel
2026-09-16 15:55:17,820 | INFO | Appel de l'API https://api.frankfurter.app/currencies
2026-09-16 15:55:18,275 | WARNING | Réponse reçue mais elle ne contient aucun taux
2026-09-16 15:55:18,277 | INFO | Fin de l'appel
2026-09-16 15:55:18,278 | INFO | Appel de l'API https://api.frankfurter.app/n-existe-pas
2026-09-16 15:55:18,717 | ERROR | Erreur HTTP 404 (Not Found)
2026-09-16 15:55:18,719 | INFO | Fin de l'appel
2026-09-16 15:55:18,720 | INFO | Appel de l'API https://domaine-inexistant.invalid/
2026-09-16 15:55:18,724 | ERROR | Serveur injoignable ou pas de connexion ([Errno 8] nodename nor servname provided, or not known)
2026-09-16 15:55:18,726 | INFO | Fin de l'appel
2026-09-16 15:55:18,727 | INFO | Appel de l'API https://example.com
2026-09-16 15:55:1

In [15]:
########  C.1  ######## 

def factorielle(n):
    """Renvoie n! de façon récursive."""
    if n <= 1:                       # cas de base, arrête la récursion
        return 1
    return n * factorielle(n - 1)    # cas récursif, la fonction s'appelle elle-même
 
 
def somme_liste(liste):
    """Renvoie la somme des éléments d'une liste de façon récursive."""
    if not liste:                              # cas de base, liste vide
        return 0
    return liste[0] + somme_liste(liste[1:])   # cas récursif, premier élément + somme du reste
 
 
print("\nfactorielle(5) =", factorielle(5))
print("somme_liste([1, 2, 3, 4]) =", somme_liste([1, 2, 3, 4]))


factorielle(5) = 120
somme_liste([1, 2, 3, 4]) = 10


In [16]:
########  C.2  ######## 

import sys
import time
from functools import lru_cache
 
VALEURS_N = (10, 15, 20, 25, 30, 35)
 
 
def mesurer_temps(fonction, n):
    """Renvoie la durée en secondes de l'appel fonction(n)."""
    debut = time.perf_counter()
    fonction(n)
    return time.perf_counter() - debut
 
 
# Version 1, récursive naïve
def fib_naif(n):
    """Renvoie le n-ième terme de Fibonacci de façon récursive naïve."""
    if n < 2:
        return n
    return fib_naif(n - 1) + fib_naif(n - 2)
 
 
print("\nFibonacci naïf")
for n in VALEURS_N:
    print(f"n = {n:>2}   {mesurer_temps(fib_naif, n):.6f} s")
 
 
# Version 2, récursive avec mémoïsation
@lru_cache(maxsize=None)
def fib_memo(n):
    """Renvoie le n-ième terme de Fibonacci en gardant les valeurs déjà calculées."""
    if n < 2:
        return n
    return fib_memo(n - 1) + fib_memo(n - 2)
 
 
print("\nFibonacci mémoïsé")
for n in VALEURS_N:
    fib_memo.cache_clear()   # vide le cache pour que chaque mesure parte de zéro
    print(f"n = {n:>2}   {mesurer_temps(fib_memo, n):.6f} s")
 
 
# Version 3, itérative
def fib_iteratif(n):
    """Renvoie le n-ième terme de Fibonacci avec une boucle."""
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a
 
 
print("\nFibonacci itératif")
for n in VALEURS_N:
    print(f"n = {n:>2}   {mesurer_temps(fib_iteratif, n):.6f} s")
 
print("\nMêmes résultats ?", fib_naif(20) == fib_memo(20) == fib_iteratif(20))
 
 
# Profondeur trop grande, RecursionError
print("\nLimite de récursion de Python", sys.getrecursionlimit())
 
try:
    factorielle(10_000)
except RecursionError:
    logger.error("factorielle(10000) dépasse la profondeur de récursion autorisée")
 
fib_memo.cache_clear()
try:
    fib_memo(10_000)
except RecursionError:
    logger.error("fib_memo(10000) dépasse la profondeur de récursion autorisée")
 
print("fib_iteratif(10000) a", len(str(fib_iteratif(10_000))), "chiffres, sans erreur")
 


Fibonacci naïf
n = 10   0.000021 s
n = 15   0.000122 s
n = 20   0.001003 s
n = 25   0.009781 s
n = 30   0.085158 s


2026-09-16 15:58:07,393 | ERROR | factorielle(10000) dépasse la profondeur de récursion autorisée
2026-09-16 15:58:07,395 | ERROR | fib_memo(10000) dépasse la profondeur de récursion autorisée


n = 35   0.828551 s

Fibonacci mémoïsé
n = 10   0.000004 s
n = 15   0.000002 s
n = 20   0.000003 s
n = 25   0.000003 s
n = 30   0.000007 s
n = 35   0.000004 s

Fibonacci itératif
n = 10   0.000002 s
n = 15   0.000001 s
n = 20   0.000001 s
n = 25   0.000001 s
n = 30   0.000001 s
n = 35   0.000001 s

Mêmes résultats ? True

Limite de récursion de Python 3000
fib_iteratif(10000) a 2090 chiffres, sans erreur


In [17]:
########  C.3  ######## 


import tracemalloc
 
 
def mesurer_memoire(fonction, n):
    """Renvoie le pic de mémoire allouée, en octets, pendant l'appel fonction(n)."""
    tracemalloc.start()
    fonction(n)
    _, pic = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return pic
 
 
versions = {"naïf": fib_naif, "mémoïsé": fib_memo, "itératif": fib_iteratif}
 
print("\nTemps en secondes")
print(f"{'n':>3} | {'appels naïf':>11} | {'naïf':>10} | {'mémoïsé':>10} | {'itératif':>10}")
for n in (10, 20, 30):
    appels = 2 * fib_iteratif(n + 1) - 1   # nombre exact d'appels faits par fib_naif(n)
    durees = []
    for fonction in versions.values():
        fib_memo.cache_clear()
        durees.append(mesurer_temps(fonction, n))
    print(f"{n:>3} | {appels:>11} | {durees[0]:>10.6f} | {durees[1]:>10.6f} | {durees[2]:>10.6f}")
 
print("\nPic de mémoire en octets")
print(f"{'n':>3} | {'naïf':>10} | {'mémoïsé':>10} | {'itératif':>10}")
for n in (10, 20, 30):
    pics = []
    for fonction in versions.values():
        fib_memo.cache_clear()
        pics.append(mesurer_memoire(fonction, n))
    print(f"{n:>3} | {pics[0]:>10} | {pics[1]:>10} | {pics[2]:>10}")
 
fib_memo.cache_clear()
fib_memo(30)
print("\nValeurs gardées en cache par fib_memo(30)", fib_memo.cache_info().currsize)
print("Profondeur de pile maximale pour n = 30   naïf 30, mémoïsé 30, itératif 1")


Temps en secondes
  n | appels naïf |       naïf |    mémoïsé |   itératif
 10 |         177 |   0.000019 |   0.000005 |   0.000001
 20 |       21891 |   0.001323 |   0.000012 |   0.000003
 30 |     2692537 |   0.084786 |   0.000030 |   0.000003

Pic de mémoire en octets
  n |       naïf |    mémoïsé |   itératif
 10 |          0 |        856 |         88
 20 |        160 |        856 |        136
 30 |        320 |       1928 |        136

Valeurs gardées en cache par fib_memo(30) 31
Profondeur de pile maximale pour n = 30   naïf 30, mémoïsé 30, itératif 1
